# From Auto-Normalization to HPO Codes

In the [quickstart](01_quickstart.ipynb), catchfly auto-normalized 197 symptom variants into 67 plain-text groups like "Abdominal pain" and "Dyspnea". That's useful — but for research you need standardized ontology codes.

**What you'll build:** The same pipeline, upgraded with:
- `ThreeStageDiscovery` (refined schema instead of single-pass)
- `SchemaOptimizer` (enriched field descriptions improve extraction)
- `CascadeNormalization` with HPO ontology (Dictionary → LLM → HPO codes with confidence scores)

**Dataset:** Same [RareArena](https://huggingface.co/datasets/THUMedInfo/RareArena) case reports, but more of them.

**Estimated API cost:** ~$0.05

In [ ]:
# pip install "catchfly[openai,medical,export,chunking]" datasets
# export OPENAI_API_KEY="sk-..."

## Why upgrade from the quickstart?

The quickstart's `LLMCanonicalization` grouped "seizures", "epileptic fits", and "convulsions" into one cluster. But it returned plain text — not an HPO code. For building patient cohorts, you need:

| Quickstart output | This notebook's output |
|---|---|
| `"Seizures"` (plain text) | `HP:0001250 — Seizure` (HPO code + confidence) |
| `"Abdominal pain"` (plain text) | `HP:0002027 — Abdominal pain` (HPO code + confidence) |
| LLMCanonicalization only | Dictionary → LLM → OntologyMapping cascade |
| SinglePassDiscovery | ThreeStageDiscovery + SchemaOptimizer |

## Step 1: Load case reports from RareArena

In [ ]:
from datasets import load_dataset

from catchfly import Document

ds = load_dataset("THUMedInfo/RareArena", split="train", streaming=True)

docs = []
for row in ds:
    case = row.get("case_report", "") or ""
    tests = row.get("test_results", "") or ""
    if len(case) < 100:
        continue
    full_text = case + (f"\n\nTest results: {tests}" if tests else "")
    docs.append(Document(content=full_text, id=row.get("_id", f"case-{len(docs)}")))
    if len(docs) >= 100:
        break

print(f"{len(docs)} case reports loaded (avg {sum(len(d.content) for d in docs) // len(docs)} chars)")
print(f"\nFirst report preview:\n{docs[0].content[:300]}...")

## Step 2: Discover the schema (ThreeStageDiscovery)

The quickstart used `SinglePassDiscovery` — one LLM call, good enough for a demo. For production, `ThreeStageDiscovery` progressively refines the schema across 3 stages with increasing sample sizes, catching fields that a single pass might miss.

In [ ]:
from catchfly.discovery import ThreeStageDiscovery

discovery = ThreeStageDiscovery(
    model="gpt-5.4-mini",
    stage1_samples=3,
    stage2_samples=5,
    stage3_samples=15,
)
schema = discovery.discover(docs, domain_hint="Rare disease clinical case reports")

print("Discovered fields:")
for name, prop in schema.json_schema["properties"].items():
    print(f"  {name}: {prop.get('type', 'object')}")

## Step 3: Enrich the schema

`SchemaOptimizer` runs extraction on test documents and asks the LLM to improve field descriptions based on errors and gaps. These enriched descriptions improve both extraction quality and downstream normalization.

In [ ]:
from catchfly.discovery import SchemaOptimizer

optimizer = SchemaOptimizer(model="gpt-5.4-mini", num_iterations=2)
enriched = optimizer.optimize(schema, test_documents=docs[:10])

# Inspect enriched field metadata — these descriptions improve extraction AND normalization
for field_name in ["symptoms", "diagnosis", "treatment"]:
    meta = enriched.field_metadata.get(field_name, {})
    if meta:
        print(f"{field_name}:")
        print(f"  Description: {meta.get('description', '(none)')}")
        print(f"  Examples: {meta.get('examples', [])}")
        print()

## Step 4: Extract structured records

With `SentenceChunking` (via chonkie), long case reports are split at sentence boundaries instead of mid-word.

In [ ]:
from catchfly.extraction import LLMDirectExtraction, SentenceChunking

extractor = LLMDirectExtraction(
    model="gpt-5.4-mini",
    chunking_strategy=SentenceChunking(chunk_size=512, overlap=64),
    on_error="collect",
)

result = extractor.extract(enriched.model, docs)

print(f"Extracted {len(result.records)} records, {len(result.errors)} errors\n")
for record in result.records[:3]:
    symptoms = getattr(record, "symptoms", []) or []
    print(f"  {getattr(record, 'diagnosis', '?')}: {symptoms[:4]}")

### Provenance tracking

Every extracted value traces back to the exact character offset in the source document:

In [ ]:
prov = result.provenance[0]
print(f"Source: {prov.source_document}")
print(f"Chars: {prov.char_start}-{prov.char_end}")
print(f"Confidence: {prov.confidence}")

## Step 5: Normalize with CascadeNormalization + HPO

This is the key upgrade from the quickstart. Instead of `LLMCanonicalization` alone, we chain three strategies:

1. **Dictionary** — known medical abbreviations (instant, zero cost)
2. **LLM** — semantic grouping of synonyms ("seizures" + "convulsions")
3. **OntologyMapping** — map to HPO codes with confidence scores

Each step handles what the previous one couldn't. The dictionary catches abbreviations, the LLM groups semantic variants, and the ontology mapper assigns standardized codes.

In [ ]:
from catchfly.normalization import CascadeNormalization

cascade = CascadeNormalization.default(
    dictionary={
        "ALT": "Alanine aminotransferase",
        "AST": "Aspartate aminotransferase",
        "CK": "Creatine kinase",
        "GFR": "Glomerular filtration rate",
        "GAGs": "Glycosaminoglycans",
    },
    model="gpt-5.4-mini",
    ontology="hpo",
)

# Collect all symptom/phenotype values across records
all_phenotypes = []
for record in result.records:
    for field in ["symptoms", "phenotypes"]:
        vals = getattr(record, field, None)
        if vals and isinstance(vals, list):
            all_phenotypes.extend(vals)

print(f"Normalizing {len(all_phenotypes)} phenotype values ({len(set(all_phenotypes))} unique)")
norm_result = cascade.normalize(all_phenotypes, context_field="symptoms")

### What each cascade step accomplished

In [ ]:
for step in norm_result.metadata["steps"]:
    print(f"Step {step['step']} ({step['strategy']}): "
          f"mapped {step['mapped_count']}, remaining {step['remaining_count']}")

### HPO codes with confidence scores

In [ ]:
# Show sample mappings with ontology IDs
sample_values = ["seizures", "hepatosplenomegaly", "dyspnea",
                 "muscle weakness", "hearing loss", "fever", "abdominal pain"]

for value in sample_values:
    if value in norm_result.mapping:
        explanation = norm_result.explain(value)
        print(f"  {explanation}")
    else:
        print(f"  {value}: (not found in extracted values)")

### All normalized clusters

In [ ]:
print(f"{norm_result.metadata['n_groups']} canonical groups:\n")
for canonical, members in sorted(norm_result.clusters.items()):
    if len(members) > 1:
        print(f"  {canonical}: {members}")

## Full pipeline (all-in-one)

Everything above in a single `Pipeline.run()` call. Here we use explicit `normalize_fields` because we're routing specific fields to a domain-specific `CascadeNormalization` with HPO ontology — the `LLMFieldSelector` from the quickstart wouldn't know to use HPO.

In [ ]:
from catchfly import Pipeline
from catchfly.discovery import ThreeStageDiscovery
from catchfly.extraction import LLMDirectExtraction, SentenceChunking
from catchfly.normalization import CascadeNormalization

pipeline = Pipeline(
    discovery=ThreeStageDiscovery(model="gpt-5.4-mini"),
    extraction=LLMDirectExtraction(
        model="gpt-5.4-mini",
        chunking_strategy=SentenceChunking(chunk_size=512),
        on_error="collect",
    ),
    normalization=CascadeNormalization.default(
        dictionary={"ALT": "Alanine aminotransferase", "AST": "Aspartate aminotransferase"},
        model="gpt-5.4-mini",
        ontology="hpo",
    ),
)

results = pipeline.run(
    docs,
    domain_hint="Rare disease clinical case reports",
    normalize_fields=["symptoms"],
    max_cost_usd=1.0,
)

print(f"Records: {len(results.records)}")
print(f"Cost: ${results.report.total_cost_usd:.3f}")
results.to_dataframe()

## What we upgraded from the quickstart

| Component | Quickstart (01) | This notebook (02) |
|---|---|---|
| Discovery | `SinglePassDiscovery` (1 LLM call) | `ThreeStageDiscovery` (3 stages, refined) |
| Schema | Raw schema | `SchemaOptimizer` enriched with examples + synonyms |
| Chunking | Default (fixed-size) | `SentenceChunking` (split at sentence boundaries) |
| Normalization | `LLMCanonicalization` (plain text groups) | `CascadeNormalization` (Dictionary → LLM → HPO codes) |
| Field selection | `LLMFieldSelector` (auto) | Explicit `normalize_fields=["symptoms"]` |
| Output | "Abdominal pain" (text) | HP:0002027 — Abdominal pain (HPO code) |

## Production tips

- **Scale up:** Replace `docs[:100]` with the full RareArena dataset or your own PMC corpus
- **Custom ontology:** `OntologyMapping(ontology="path/to/custom_terms.csv")`
- **Checkpoint/resume:** Add `checkpoint_dir="./checkpoints"` for large corpora
- **Schema review:** Use `on_schema_ready` callback to inspect/modify before extraction
- **Cost budget:** Set `max_cost_usd=5.0` to prevent runaway spending
- **Ground truth:** RareArena includes `diagnosis` and `Orpha_id` — use them to evaluate extraction accuracy